[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/35_reinforcement_learning/03_q_learning_and_sarsa.ipynb)

# Bài 03 — Q-learning và SARSA

**🎯 Mục tiêu**
- Từ **dự đoán** sang **điều khiển**: học hàm $Q(s, a)$ và cải thiện chính sách không cần mô hình.
- **SARSA** (on-policy) vs **Q-learning** (off-policy): cập nhật khác một chữ, hành vi khác hẳn — thí nghiệm Cliff Walking.
- Q-learning tìm chính sách (gần) tối ưu trên FrozenLake trơn — so với lời giải quy hoạch động.
- **Thiên lệch cực đại** (maximization bias) và **Double Q-learning**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/35_reinforcement_learning

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.envs import ACTION_NAMES, CliffWalking, GridWorld
from ml_course.visualization import set_style

set_style()

## 1. Học $Q(s, a)$ để điều khiển

Biết $V^\pi$ chưa đủ để chọn hành động khi không có mô hình (phải biết hành động dẫn tới đâu). Học thẳng **giá trị hành động** $Q(s, a)$ — return kỳ vọng nếu làm $a$ ở $s$ rồi theo chính sách — thì chính sách tốt hơn chỉ là $\arg\max_a Q(s, a)$. Luân phiên "ước lượng $Q$ ↔ hành động tham lam theo $Q$" (generalized policy iteration), kèm **ε-greedy** để vẫn khám phá.

Hai cách cập nhật TD cho $Q$ sau bước $(s, a, r, s')$:

$$\textbf{SARSA:}\quad Q(s, a) \leftarrow Q(s, a) + \alpha\left[r + \gamma Q(s', a') - Q(s, a)\right], \quad a' \text{ là hành động sẽ thực sự làm (ε-greedy)}$$
$$\textbf{Q-learning:}\quad Q(s, a) \leftarrow Q(s, a) + \alpha\left[r + \gamma\max_{a''}Q(s', a'') - Q(s, a)\right]$$

- SARSA (**on-policy**) học giá trị của **chính chính sách đang chạy** (kể cả các bước khám phá ngẫu nhiên).
- Q-learning (**off-policy**) học giá trị của chính sách **tham lam tối ưu**, bất kể đang hành động thế nào — có thể học từ dữ liệu do chính sách khác tạo ra (nền tảng của DQN, bài 04).

In [ ]:
def epsilon_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(Q.shape[1]))
    return int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))         # phá hoà ngẫu nhiên


def td_control(env, method, episodes, alpha, eps, gamma=1.0, seed=0, eps_decay_to=None):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for ep in range(episodes):
        e = eps if eps_decay_to is None else max(eps_decay_to, eps * (1 - ep / (episodes / 2)))
        s = env.reset(seed=seed * 1_000_000 + ep)
        a = epsilon_greedy(Q, s, e, rng)
        done, total = False, 0.0
        while not done:
            s2, r, done = env.step(a)
            total += r
            a2 = epsilon_greedy(Q, s2, e, rng)
            if done:
                target = r
            elif method == "SARSA":
                target = r + gamma * Q[s2, a2]
            else:
                target = r + gamma * Q[s2].max()
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = s2, a2
        returns.append(total)
    return Q, np.array(returns)

## 2. Cliff Walking: an toàn hay tối ưu?

Lưới 4×12: đi từ S (góc dưới trái) tới G (góc dưới phải); mỗi bước −1; bước xuống **vực** (hàng dưới giữa S và G) bị −100 và quay về S. Đường tối ưu: men sát mép vực (13 bước). Cả hai thuật toán dùng ε = 0.1 cố định, α = 0.5; 500 lượt, 20 seed.

In [ ]:
cliff = CliffWalking()
results = {}
t0 = time.perf_counter()
for method in ["SARSA", "Q-learning"]:
    runs = [td_control(cliff, method, episodes=500, alpha=0.5, eps=0.1, seed=s) for s in range(20)]
    results[method] = (runs[0][0], np.array([r for _, r in runs]))
print(f"{time.perf_counter() - t0:.0f}s")

fig, ax = plt.subplots(figsize=(7.5, 4))
for method, (_, rets) in results.items():
    smooth = np.convolve(rets.mean(0), np.ones(10) / 10, mode="valid")
    ax.plot(smooth, label=method)
    print(f"{method:10s}: tổng thưởng mỗi lượt KHI ĐANG HỌC (100 lượt cuối, TB 20 seed) = {rets[:, -100:].mean():.1f}")
ax.set_ylim(-120, 0)
ax.set_xlabel("lượt")
ax.set_ylabel("tổng thưởng mỗi lượt (ε-greedy)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def greedy_path(env, Q, max_steps=100):
    s, done, path, total = env.reset(), False, [env.start], 0.0
    while not done and len(path) <= max_steps:
        s, r, done = env.step(int(Q[s].argmax()))
        path.append(s)
        total += r
    return path, total


fig, axes = plt.subplots(1, 2, figsize=(13, 2.8))
for ax, (method, (Q, _)) in zip(axes, results.items()):
    path, total = greedy_path(cliff, Q)
    grid = np.zeros((4, 12))
    grid[3, 1:11] = -1
    ax.imshow(grid, cmap="Reds_r", vmin=-1.5, vmax=0)
    rows, cols = np.divmod(path, 12)
    ax.plot(cols, rows, "o-", color="C0")
    ax.set_title(f"{method}: chính sách tham lam — {len(path) - 1} bước, tổng thưởng {total:.0f}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

- **Q-learning** học được đường **tối ưu** (sát vực, 13 bước) — nhưng trong lúc học, vì vẫn khám phá ε-greedy, thỉnh thoảng bước nhầm xuống vực → tổng thưởng khi học **thấp hơn**.
- **SARSA** học giá trị của chính sách *có khám phá*: đi sát vực thì 10% bước ngẫu nhiên có thể rơi xuống → giá trị thấp → chọn đường **xa vực**, dài hơn nhưng an toàn — tổng thưởng khi học **cao hơn**.

Không cái nào "đúng hơn": nếu sau khi học ta tắt khám phá, đường của Q-learning tốt hơn; nếu tác tử phải **hành động thật trong lúc học** (robot đắt tiền, hệ thống đang chạy), SARSA thận trọng hơn. Nếu giảm dần ε về 0, cả hai hội tụ về chính sách tối ưu.

## 3. Q-learning trên FrozenLake trơn

Môi trường **ngẫu nhiên** (trượt), thưởng **thưa** (chỉ 1 khi tới đích). Q-learning 20 000 lượt, ε giảm từ 1 về 0.05, α = 0.1, γ = 0.99 — so với $Q^*$ từ value iteration (bài 01).

In [ ]:
lake = GridWorld("4x4", slip=2 / 3)
GAMMA = 0.99
V = np.zeros(16)
for _ in range(2000):                                                  # value iteration (bài 01)
    Q_star = np.array([[sum(p * (r + GAMMA * V[s2] * (not d)) for p, s2, r, d in lake.P[s][a]) for a in range(4)] for s in range(16)])
    V = Q_star.max(1)

t0 = time.perf_counter()
Q_lake, rets = td_control(lake, "Q-learning", episodes=20000, alpha=0.1, eps=1.0, gamma=GAMMA, seed=0, eps_decay_to=0.05)
print(f"{time.perf_counter() - t0:.0f}s | tỉ lệ thắng khi học (1 000 lượt cuối, còn ε = 0.05) = {rets[-1000:].mean():.0%}")


def win_rate(env, pi, episodes=2000):
    wins = 0
    for ep in range(episodes):
        s, done = env.reset(seed=50_000 + ep), False
        while not done:
            s, r, done = env.step(pi[s])
        wins += r
    return wins / episodes


non_terminal = [s for s in range(16) if s not in lake.terminal]
print(f"chính sách tham lam từ Q-learning: thắng {win_rate(lake, Q_lake.argmax(1)):.1%} | tối ưu (quy hoạch động): {win_rate(lake, Q_star.argmax(1)):.1%}")
print(f"trùng hành động tối ưu ở {np.mean(Q_lake.argmax(1)[non_terminal] == Q_star.argmax(1)[non_terminal]):.0%} ô | "
      f"sai lệch |Q − Q*| lớn nhất = {np.abs(Q_lake - Q_star)[non_terminal].max():.2f}")

Không biết gì về xác suất trượt, Q-learning đạt tỉ lệ thắng gần bằng chính sách tối ưu. Giá trị $Q$ vẫn sai lệch đáng kể (learning rate cố định → ước lượng dao động theo nhiễu của môi trường), nhưng **thứ tự** giữa các hành động — thứ quyết định chính sách — đã gần đúng. Cái giá: hàng chục nghìn lượt chơi cho 16 trạng thái — RL dạng bảng rất tốn mẫu.

## 4. Thiên lệch cực đại và Double Q-learning

Q-learning dùng $\max_a Q(s', a)$ của các ước lượng **nhiễu** → $\mathbb E[\max_a \hat Q] \ge \max_a\mathbb E[\hat Q]$: luôn **đánh giá quá cao**. Ví dụ của Sutton & Barto (6.7): từ A, đi phải → kết thúc (thưởng 0); đi trái → B, từ B có 10 hành động đều kết thúc với thưởng $\sim\mathcal N(-0.1, 1)$. Đi trái **tệ hơn** (kỳ vọng −0.1), nhưng max của 10 ước lượng nhiễu thường dương → Q-learning bị "dụ" đi trái.

**Double Q-learning** (van Hasselt, 2010): giữ hai bảng $Q_1, Q_2$; một bảng **chọn** hành động, bảng kia **đánh giá**: $Q_1(s, a) \leftarrow Q_1(s, a) + \alpha[r + \gamma Q_2(s', \arg\max_{a'}Q_1(s', a')) - Q_1(s, a)]$ (và đổi vai ngẫu nhiên).

In [ ]:
def maximization_bias_experiment(double, runs=1000, episodes=300, alpha=0.1, eps=0.1, n_b_actions=10):
    left_counts = np.zeros(episodes)
    for run in range(runs):
        rng = np.random.default_rng(run)
        QA = [np.zeros(2), np.zeros(2)]                                # A: 0 = trái (sang B), 1 = phải (kết thúc)
        QB = [np.zeros(n_b_actions), np.zeros(n_b_actions)]
        for ep in range(episodes):
            qa = QA[0] + QA[1] if double else QA[0]
            a = int(rng.integers(2)) if rng.random() < eps else int(rng.choice(np.flatnonzero(qa == qa.max())))
            left_counts[ep] += a == 0
            if a == 1:
                i = int(rng.integers(2)) if double else 0
                QA[i][1] += alpha * (0 - QA[i][1])
                continue
            i = int(rng.integers(2)) if double else 0                  # bảng được cập nhật
            j = 1 - i if double else 0                                 # bảng dùng để đánh giá
            QA[i][0] += alpha * (QB[j][int(QB[i].argmax())] - QA[i][0]) if double else alpha * (QB[0].max() - QA[0][0])
            qb = QB[0] + QB[1] if double else QB[0]
            b = int(rng.integers(n_b_actions)) if rng.random() < eps else int(rng.choice(np.flatnonzero(qb == qb.max())))
            r = rng.normal(-0.1, 1)
            k = int(rng.integers(2)) if double else 0
            QB[k][b] += alpha * (r - QB[k][b])
    return left_counts / runs


t0 = time.perf_counter()
fig, ax = plt.subplots(figsize=(7, 3.8))
for name, double in [("Q-learning", False), ("Double Q-learning", True)]:
    left = maximization_bias_experiment(double)
    ax.plot(left, label=name)
    print(f"{name:18s}: % đi trái (tệ) ở 20 lượt đầu = {left[:20].mean():.0%}, 50 lượt cuối = {left[-50:].mean():.0%}")
ax.axhline(0.05, color="gray", ls=":", label="tối ưu (chỉ do ε = 0.1)")
ax.set_xlabel("lượt")
ax.set_ylabel("% chọn đi trái từ A")
ax.legend()
plt.tight_layout()
plt.show()
print(f"{time.perf_counter() - t0:.0f}s")

Q-learning bị dụ đi trái rất nhiều trong giai đoạn đầu và đến cuối vẫn đi trái nhiều hơn mức tối ưu; Double Q-learning bị dụ ít hơn hẳn ngay từ đầu và nhanh chóng về gần mức tối ưu (5% — chỉ do khám phá ngẫu nhiên). Thiên lệch này càng nghiêm trọng khi nhiều hành động và ước lượng nhiễu — chính tình huống của DQN với mạng nơ-ron; **Double DQN** (bài 04) áp dụng đúng ý tưởng này.

## 5. Giới hạn của phương pháp dạng bảng

Bảng $Q$ có $|\mathcal S|\times|\mathcal A|$ ô, mỗi ô phải được thăm nhiều lần, và không **tổng quát hoá**: học ở ô này không giúp gì ô bên cạnh. Với trạng thái liên tục (vị trí, vận tốc của CartPole) hay ảnh, cần **xấp xỉ hàm** $Q_\theta(s, a)$ bằng mạng nơ-ron — bài 04.

## 🧠 Kiểm tra nhanh

1. Viết cập nhật SARSA và Q-learning. Khác nhau ở đâu?
2. On-policy và off-policy là gì?
3. Vì sao trong Cliff Walking, SARSA chọn đường xa vực còn Q-learning đi sát vực? Thuật toán nào có tổng thưởng khi học cao hơn?
4. Thiên lệch cực đại là gì? Double Q-learning khắc phục ra sao?
5. Vì sao RL dạng bảng không mở rộng được?

## 📌 Tóm tắt

- Điều khiển không mô hình: học $Q(s, a)$ bằng TD + ε-greedy; SARSA (on-policy, giá trị của chính sách đang chạy) vs Q-learning (off-policy, giá trị của chính sách tối ưu).
- Cliff Walking: Q-learning học đường tối ưu nhưng rủi ro khi đang học; SARSA an toàn hơn.
- Q-learning đạt gần tối ưu trên môi trường ngẫu nhiên nhưng tốn nhiều mẫu; max trên ước lượng nhiễu gây đánh giá quá cao → Double Q-learning.

## 📚 Tài liệu tham khảo

- Sutton & Barto (2018) — chương 6 (SARSA, Q-learning, maximization bias, Double Q-learning)
- Watkins & Dayan (1992) — *Q-learning*
- van Hasselt (2010) — *Double Q-learning*
- Rummery & Niranjan (1994) — *On-line Q-learning using connectionist systems* (SARSA)